In [ ]:
# Caso necessário, instale as dependências uma única vez no ambiente:
# %pip install requests pandas tqdm

import io
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import requests
from tqdm.auto import tqdm
from IPython.display import display

# Download automático é opcional e só é usado quando o notebook roda no Colab.
try:
    from google.colab import files as colab_files
except ImportError:
    colab_files = None

# ============================================================
# CONFIGURAÇÕES DO PROJETO
# ============================================================
ANO_INICIAL = 2020
ANO_FINAL = pd.Timestamp.today().year
DATA_HOJE = pd.Timestamp.today().normalize()
UF_ALVO = "SP"
MAX_WORKERS = 8

ARQUIVO_FINAL = "Base_Dengue_SP_2020_ate_hoje.csv"
ARQUIVO_ERROS = "Municipios_SP_com_erro_InfoDengue.csv"

API_IBGE = "https://servicodados.ibge.gov.br/api/v1/localidades/municipios"
API_INFODENGUE = "https://info.dengue.mat.br/api/alertcity"

# ============================================================
# TESTE DE CONEXÃO
# ============================================================
teste_params = {
    "geocode": 3550308,  # Município de São Paulo
    "disease": "dengue",
    "format": "csv",
    "ew_start": 1,
    "ew_end": 53,
    "ey_start": ANO_FINAL,
    "ey_end": ANO_FINAL,
}

try:
    teste = requests.get(API_INFODENGUE, params=teste_params, timeout=60)
    print("Status da API do InfoDengue:", teste.status_code)
except Exception as erro:
    print("Erro ao testar a API do InfoDengue:", erro)

# ============================================================
# MUNICÍPIOS DO ESTADO DE SÃO PAULO — IBGE
# ============================================================
resposta_ibge = requests.get(API_IBGE, timeout=60)
resposta_ibge.raise_for_status()

municipios = []
for municipio in resposta_ibge.json():
    try:
        uf = municipio["microrregiao"]["mesorregiao"]["UF"]["sigla"]
        if uf == UF_ALVO:
            municipios.append({
                "geocode": int(municipio["id"]),
                "municipio": municipio["nome"],
                "UF": uf,
            })
    except (KeyError, TypeError):
        continue

df_municipios = (
    pd.DataFrame(municipios)
    .drop_duplicates(subset=["geocode"])
    .sort_values("municipio")
    .reset_index(drop=True)
)

print("Municípios de SP encontrados:", len(df_municipios))

# ============================================================
# CONSULTA AO INFODENGUE
# ============================================================
def consultar_municipio(registro):
    geocode = registro["geocode"]
    municipio = registro["municipio"]
    uf = registro["UF"]

    resultados = []
    erros_local = []

    for ano in range(ANO_INICIAL, ANO_FINAL + 1):
        parametros = {
            "geocode": geocode,
            "disease": "dengue",
            "format": "csv",
            "ew_start": 1,
            "ew_end": 53,
            "ey_start": ano,
            "ey_end": ano,
        }

        try:
            resposta = requests.get(
                API_INFODENGUE,
                params=parametros,
                timeout=90,
            )

            if resposta.status_code != 200:
                erros_local.append(f"Ano {ano}: HTTP {resposta.status_code}")
                continue

            texto = resposta.text.strip()
            if not texto:
                continue
            if texto.startswith("[EE]"):
                erros_local.append(f"Ano {ano}: {texto}")
                continue

            try:
                dados = pd.read_csv(io.StringIO(texto))
            except Exception as erro_csv:
                erros_local.append(f"Ano {ano}: {erro_csv}")
                continue

            if dados.empty:
                continue

            dados["geocode"] = geocode
            dados["municipio_nome"] = municipio
            dados["UF"] = uf

            if "data_iniSE" in dados.columns:
                # A API retorna data no padrão ISO. Recortamos os 10 primeiros
                # caracteres para manter a conversão explícita e previsível.
                dados["data_iniSE"] = pd.to_datetime(
                    dados["data_iniSE"].astype("string").str[:10],
                    format="%Y-%m-%d",
                    errors="coerce",
                )
                dados = dados[dados["data_iniSE"].notna()].copy()
                dados = dados[
                    (dados["data_iniSE"].dt.year >= ANO_INICIAL)
                    & (dados["data_iniSE"] <= DATA_HOJE)
                ].copy()

            if not dados.empty:
                resultados.append(dados)

        except Exception as erro:
            erros_local.append(f"Ano {ano}: {repr(erro)}")

    if not resultados:
        return {
            "status": "sem_dados",
            "geocode": geocode,
            "municipio": municipio,
            "UF": uf,
            "dados": None,
            "erros": erros_local,
        }

    try:
        dados_final = pd.concat(resultados, ignore_index=True)
    except Exception as erro:
        return {
            "status": "erro",
            "geocode": geocode,
            "municipio": municipio,
            "UF": uf,
            "dados": None,
            "erros": [repr(erro)],
        }

    return {
        "status": "ok",
        "geocode": geocode,
        "municipio": municipio,
        "UF": uf,
        "dados": dados_final,
        "erros": erros_local,
    }


inicio = time.time()
dados_finais = []
municipios_sem_dados = []
municipios_com_erro = []

lista_municipios = df_municipios.to_dict("records")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    tarefas = {
        executor.submit(consultar_municipio, municipio): municipio
        for municipio in lista_municipios
    }

    for tarefa in tqdm(
        as_completed(tarefas),
        total=len(lista_municipios),
        desc="Consultando municípios de SP",
    ):
        try:
            resultado = tarefa.result()
        except Exception as erro:
            municipio = tarefas[tarefa]
            municipios_com_erro.append({
                "geocode": municipio["geocode"],
                "municipio": municipio["municipio"],
                "UF": municipio["UF"],
                "erro": repr(erro),
            })
            continue

        if resultado["status"] == "ok":
            dados_finais.append(resultado["dados"])
        elif resultado["status"] == "sem_dados":
            municipios_sem_dados.append({
                "geocode": resultado["geocode"],
                "municipio": resultado["municipio"],
                "UF": resultado["UF"],
                "erro": "Nenhum dado retornado",
            })
        else:
            municipios_com_erro.append({
                "geocode": resultado["geocode"],
                "municipio": resultado["municipio"],
                "UF": resultado["UF"],
                "erro": " | ".join(resultado.get("erros", [])),
            })

if not dados_finais:
    raise RuntimeError("A API não retornou dados para os municípios de São Paulo.")

# ============================================================
# CONSOLIDAÇÃO E QUALIDADE
# ============================================================
df_sp = pd.concat(dados_finais, ignore_index=True)

if "data_iniSE" in df_sp.columns:
    df_sp["data_iniSE"] = pd.to_datetime(
        df_sp["data_iniSE"],
        errors="coerce",
    )
    df_sp = df_sp[df_sp["data_iniSE"].notna()].copy()
    df_sp = df_sp[df_sp["data_iniSE"] <= DATA_HOJE].copy()

# Reforça o recorte de SP.
if "UF" in df_sp.columns:
    df_sp = df_sp[df_sp["UF"].astype(str).str.upper().eq(UF_ALVO)].copy()

# Uma observação deve ser única por município e início da semana epidemiológica.
if {"geocode", "data_iniSE"}.issubset(df_sp.columns):
    df_sp = df_sp.drop_duplicates(
        subset=["geocode", "data_iniSE"],
        keep="last",
    )
else:
    df_sp = df_sp.drop_duplicates()

colunas_ordenacao = [
    c for c in ["municipio_nome", "data_iniSE"]
    if c in df_sp.columns
]
if colunas_ordenacao:
    df_sp = df_sp.sort_values(colunas_ordenacao, kind="stable")

colunas_inicio = ["UF", "municipio_nome", "geocode", "data_iniSE", "SE"]
colunas_existentes = [c for c in colunas_inicio if c in df_sp.columns]
outras_colunas = [c for c in df_sp.columns if c not in colunas_existentes]
df_sp = df_sp[colunas_existentes + outras_colunas].reset_index(drop=True)

# O CSV é salvo explicitamente em ISO (AAAA-MM-DD), compatível com o notebook principal.
df_sp.to_csv(
    ARQUIVO_FINAL,
    index=False,
    encoding="utf-8-sig",
    sep=",",
    date_format="%Y-%m-%d",
)

# ============================================================
# REGISTRO DE ERROS E RESUMO
# ============================================================
todos_erros = municipios_sem_dados + municipios_com_erro
if todos_erros:
    pd.DataFrame(todos_erros).to_csv(
        ARQUIVO_ERROS,
        index=False,
        encoding="utf-8-sig",
    )

tempo_total = (time.time() - inicio) / 60

print("Arquivo:", ARQUIVO_FINAL)
print("Registros:", len(df_sp))
print("Municípios:", df_sp["municipio_nome"].nunique() if "municipio_nome" in df_sp.columns else "n/d")
print("Estados:", df_sp["UF"].nunique() if "UF" in df_sp.columns else "n/d")
print("Municípios sem dados:", len(municipios_sem_dados))
print("Municípios com erro:", len(municipios_com_erro))
print("Tempo total:", f"{tempo_total:.1f} minutos")

display(df_sp.head(20))

if colab_files is not None:
    colab_files.download(ARQUIVO_FINAL)
    if todos_erros:
        colab_files.download(ARQUIVO_ERROS)
else:
    print("Execução fora do Colab: os arquivos foram salvos na pasta atual.")
